# 의견제출통지서 PDF 이미지 추출

**목적**: 특허청이 선행디자인/공보/공지 디자인을 이유로 거절한 사례 수집  
**대상 키워드**: `디자인보호법 제33조`, `디자인보호법 제33조제2항`, `용이 창작 디자인`, `참증디자인`  
**동작 흐름**:
1. API로 pdf 다운로드
2. PDF 텍스트에서 키워드 탐색 (페이지 단위)
3. 1개 이상 키워드 매칭 시 → 전체 이미지 추출
4. 결과 요약 CSV 저장 + 이미지 미리보기

In [ ]:
import requests
import pandas as pd
from pathlib import Path
from pypdf import PdfReader
from collections import defaultdict
import os


In [ ]:
import pandas as pd

# 거절 디자인건 출원번호 엑셀 파일 로드
excel_file = r'/Users/nanahyun/Documents/GitHub/final_develop/design/data/KIPRIS_xlsx/거절건 출원번호.xlsx'
df_excel = pd.read_excel(excel_file, header=None, engine='calamine')

# 9행부터 C열(index 2) 출원번호 추출
application_numbers = []
for val in df_excel.iloc[8:, 2]:
    if pd.notna(val) and str(val).strip():
        application_numbers.append(str(val).strip())

print(f"📊 {excel_file} 파일에서 {len(application_numbers)}개의 출원번호를 읽었습니다.\n")

In [ ]:
import time
API_KEY = os.getenv("KIPRISPLUS_API_KEY")

# 의견제출통지서 API 호출
base_url = "http://plus.kipris.or.kr/openapi/rest/IntermediateDocumentOPService/advancedSearchInfo"

# api 호출 결과 저장할 폴더
os.makedirs("../data", exist_ok=True)

success_count = 0
fail_count = 0

for idx, app_num in enumerate(application_numbers, 1):
    try:
        # = 기호 URL 인코딩 방지를 위해 URL에 직접 삽입
        url = f"{base_url}?accessKey={API_KEY}&applicationNumber={app_num}"
        
        t0 = time.time()
        r = requests.get(url, timeout=20)
        latency_ms = int((time.time() - t0) * 1000)
        
        print(f"[{idx}/{len(application_numbers)}] {app_num} - status: {r.status_code}, latency: {latency_ms}ms")
        
        if r.status_code == 200:
            file_path = f"../data/{app_num}.xml"
            with open(file_path, 'w', encoding='utf-8') as f:
                f.write(r.text)
            print(f"         ✅ 저장: {file_path}")
            success_count += 1
        else:
            print(f"         ❌ 오류: {r.status_code} - {r.text[:100]}")
            fail_count += 1
            
        time.sleep(0.5)
        
    except requests.exceptions.RequestException as e:
        print(f"[{idx}/{len(application_numbers)}] {app_num} - Request failed: {type(e).__name__}")
        fail_count += 1

print(f"\n✅ 완료: {success_count}개 저장, {fail_count}개 실패")

In [ ]:
import glob
from IPython.display import display, Image as IPImage

image_files = sorted(
    glob.glob(str(Path(OUTPUT_DIR) / "**" / "*.jpg"), recursive=True) +
    glob.glob(str(Path(OUTPUT_DIR) / "**" / "*.png"), recursive=True)
)

print(f"전체 이미지: {len(image_files)}개 | 미리보기: {PREVIEW_COUNT}개")
for img_path in image_files[:PREVIEW_COUNT]:
    print(f"\n📄 {img_path}")
    display(IPImage(filename=img_path, width=500))

In [ ]:
print("=" * 50)
print("[ 수집 결과 요약 ]")
print(f"  전체 PDF           : {len(downloaded)}개")
print(f"  키워드 매칭 PDF    : {len(matched)}개")
print(f"  추출된 이미지      : {total_images}개")
print(f"  결과 CSV           : {RESULT_CSV}")
print(f"  이미지 저장 위치   : {Path(OUTPUT_DIR).resolve()}")
print()
print("[ 키워드별 매칭 통계 ]")
for kw in KEYWORDS:
    hit_count = df[df[kw] != ""].shape[0]
    print(f"  {kw:20s} : {hit_count}건")
print("=" * 50)